In [2]:
import pandas as pd

master_products = pd.read_csv("../data/phase1/master_products.csv")

interactions = pd.read_csv("../data/phase2/customer_product_interactions.csv")

content_products = pd.read_csv(
    "../models/content_based/content_products.csv"
)
collab_recommendations = pd.read_csv(
    "../models/collaborative_candidates_C00001.csv"
)

In [3]:
content_products.columns.tolist()

['product_id',
 'product_name',
 'category',
 'subcategory',
 'brand',
 'product_type',
 'description',
 'combined_features']

In [4]:
import os
os.listdir("../models/content_based")

['content_products.csv', 'tfidf_matrix.npz', 'tfidf_vectorizer.pkl']

In [5]:
import joblib
from scipy.sparse import load_npz

tfidf_matrix = load_npz(
    "../models/content_based/tfidf_matrix.npz"
)

tfidf = joblib.load(
    "../models/content_based/tfidf_vectorizer.pkl"
)

In [6]:
from sklearn.metrics.pairwise import cosine_similarity
customer_id = "C00001"

purchased_products = interactions[
    interactions["customer_id"] == customer_id
]["product_id"].unique()
purchased_indices = content_products[
    content_products["product_id"].isin(purchased_products)
].index
candidate_indices = content_products[
    content_products["product_id"].isin(collab_recommendations["product_id"])
].index

content_scores = cosine_similarity(
    tfidf_matrix[candidate_indices],
    tfidf_matrix[purchased_indices]
).max(axis=1)

collab_recommendations["content_score"] = content_scores
collab_recommendations.head(10)

,product_id,collaborative_score,product_name,category,brand,price,rating,content_score
0,P026388,0.155126,"Shower Sponge - Lemon & White, Colour May Vary",Beauty & Hygiene,Panache,159.2,4.0,0.118010
1,P010637,0.103418,Water Juice Glass Set - Salsa Hi Ball,"Kitchen, Garden & Pets",Ocean,679.0,3.0,0.143938
2,P014337,0.103418,Bio Aloevera Lotion - 30 Spf Sunscreen For Nor...,Beauty & Hygiene,BIOTIQUE,188.0,4.0,0.061709
3,P026457,0.103418,Pickle - Garlic,Snacks & Branded Foods,Nirapara,135.0,3.7,0.124596
4,P017394,0.103418,So Soft 3 Ply - Toilet Tissue Rolls,Cleaning & Household,Origami,310.0,4.2,0.138429
5,P011377,0.103418,"Dry Dog Food - PRO, Expert Nutrition for Activ...","Kitchen, Garden & Pets",Pedigree,2700.0,4.8,0.203335
6,P022821,0.103418,Large Period Cup For Medium Or Heavy Flow - Po...,Beauty & Hygiene,ezy,439.2,3.7,0.078531
7,P000801,0.051709,Premium - 3 Follow-Up Formula,Baby Care,Dexolac,595.0,4.5,0.417582
8,P019543,0.051709,D-Tan Scrub For Men,Beauty & Hygiene,QRAA,263.0,4.2,0.064228
9,P022417,0.051709,Chalk Gold - For Cockroaches,Cleaning & Household,Krazy Lines,25.0,3.6,0.031126


In [7]:
collab_recommendations["hybrid_score"] = (
    0.6 * collab_recommendations["collaborative_score"] + 0.4 * collab_recommendations["content_score"]
)
collab_recommendations = collab_recommendations.sort_values(
    "hybrid_score", ascending=False
)
collab_recommendations.head(10)

,product_id,collaborative_score,product_name,category,brand,price,rating,content_score,hybrid_score
7,P000801,0.051709,Premium - 3 Follow-Up Formula,Baby Care,Dexolac,595.0,4.5,0.417582,0.198058
15,P026869,0.051709,Anti-Ageing Night Cream,Beauty & Hygiene,INATUR,337.5,1.0,0.321588,0.159661
31,P026894,0.051709,Vitamin C Foaming Brightening Face Wash - With...,Beauty & Hygiene,StBotanica,445.0,4.7,0.292085,0.147859
5,P011377,0.103418,"Dry Dog Food - PRO, Expert Nutrition for Activ...","Kitchen, Garden & Pets",Pedigree,2700.0,4.8,0.203335,0.143385
0,P026388,0.155126,"Shower Sponge - Lemon & White, Colour May Vary",Beauty & Hygiene,Panache,159.2,4.0,0.118010,0.140280
29,P003744,0.051709,Pizza Cutter - Royal,"Kitchen, Garden & Pets",Anjali,50.0,3.9,0.271730,0.139717
12,P023445,0.051709,FIAMA Shower Gel Lemongrass Jojoba 100ml + FIA...,Beauty & Hygiene,Fiama,108.9,NaN,0.258456,0.134408
1,P010637,0.103418,Water Juice Glass Set - Salsa Hi Ball,"Kitchen, Garden & Pets",Ocean,679.0,3.0,0.143938,0.119626
4,P017394,0.103418,So Soft 3 Ply - Toilet Tissue Rolls,Cleaning & Household,Origami,310.0,4.2,0.138429,0.117422
11,P016052,0.051709,Frankincense Essential Oil,Beauty & Hygiene,Nectar Valley,465.0,3.0,0.212628,0.116076


In [8]:
interaction_matrix = interactions.pivot_table(
    index = "customer_id",
    columns = "product_id", 
    values = "total_quantity",
    aggfunc = "sum",
    fill_value=0
)
print(interaction_matrix.shape)

(3000, 27145)


In [9]:
binary_interaction = (interaction_matrix>0).astype(int)
print("Binary interaction matrix shape:", binary_interaction.shape)

Binary interaction matrix shape: (3000, 27145)


In [10]:
from sklearn.metrics.pairwise import cosine_similarity
customer_similarity = cosine_similarity(binary_interaction)
print("Customer similarity matrix shape:", customer_similarity.shape)

Customer similarity matrix shape: (3000, 3000)


In [11]:
import numpy as np
def collaborative_candidates(customer_id, n_similar=20, n_candidates=50):
    customer_index=interaction_matrix.index.get_loc(customer_id)
    similarity_scores = customer_similarity[customer_index]
    similar_indices = np.argsort(similarity_scores)[::-1]
    candidates={}
    processed_similar = 0
    for i in similar_indices:
        similar_customers = interaction_matrix.index[i]
        if similar_customers == customer_id:
            continue
        similarity = similarity_scores[i]
        customer_products = interaction_matrix.loc[similar_customers]
        purchased_products = customer_products[customer_products>0]
        for product_id, quantity in purchased_products.items():
            if interaction_matrix.loc[customer_id, product_id]>0:
                continue
            score = similarity*quantity
            candidates[product_id] = (candidates.get(product_id, 0) + score)
        processed_similar+=1
        if processed_similar >= n_similar:
            break    
        
    recommendations = (
        pd.DataFrame(
            list(candidates.items()),
            columns=["product_id", "collaborative_score"]
        )
        .sort_values(
            "collaborative_score",
            ascending=False
        )
        .head(n_candidates)
    )
    return recommendations

In [12]:
collab_test = collaborative_candidates(
    "C00001", n_similar=20, n_candidates=50
) 
collab_test.head(10)

,product_id,collaborative_score
91,P015536,0.190693
26,P026388,0.155126
10,P010637,0.151091
33,P004097,0.148704
54,P016702,0.148704
67,P000237,0.143019
70,P002553,0.143019
102,P026493,0.143019
97,P021849,0.143019
88,P012599,0.143019


In [13]:
def hybrid_recommend(customer_id, n=10):
    purchased_products = interactions[
        interactions["customer_id"] == customer_id
    ]["product_id"].unique()

    collab_candidates = collaborative_candidates(
        customer_id, n_similar=20, n_candidates=50
    )

    collab_candidates = collab_candidates.merge(
        master_products[
            [
                "product_id",
                "product_name",
                "category",
                "brand",
                "price",
                "rating"
            ]
        ],
        on="product_id",
        how="left"
    )

    product_to_index = pd.Series(
        content_products.index,
        index=content_products["product_id"]
    )

    purchased_indices = [
        product_to_index[p]
        for p in purchased_products
        if p in product_to_index.index
    ]

    candidate_indices = [
        product_to_index[p]
        for p in collab_candidates["product_id"]
        if p in product_to_index.index
    ]

    if purchased_indices and candidate_indices:
        content_scores = cosine_similarity(
            tfidf_matrix[candidate_indices],
            tfidf_matrix[purchased_indices]
        ).max(axis=1)
    else:
        content_scores = np.zeros(len(collab_candidates))

    collab_candidates["content_scores"] = content_scores

    from sklearn.preprocessing import MinMaxScaler

    scaler = MinMaxScaler()

    collab_candidates[
        ["collaborative_score", "content_scores"]
    ] = scaler.fit_transform(
        collab_candidates[
            ["collaborative_score", "content_scores"]
        ]
    )

    collab_candidates["hybrid_score"] = (
        0.6 * collab_candidates["collaborative_score"]
        + 0.4 * collab_candidates["content_scores"]
    )

    recommendations = collab_candidates.sort_values(
        "hybrid_score",
        ascending=False
    ).head(n)

    return recommendations[
        [
            "product_id",
            "product_name",
            "category",
            "brand",
            "price",
            "rating",
            "collaborative_score",
            "content_scores",
            "hybrid_score"
        ]
    ]


In [19]:
all_recommendations = []

for customer_id in interactions["customer_id"].unique():
    customer_recommendations = hybrid_recommend(customer_id, n=10)

    customer_recommendations["customer_id"] = customer_id
    all_recommendations.append(customer_recommendations)
    print("Processed:", customer_id)
print("customers processed:", len(all_recommendations))    

Processed: C00001
Processed: C00002
Processed: C00003
Processed: C00004
Processed: C00005
Processed: C00006
Processed: C00007
Processed: C00008
Processed: C00009
Processed: C00010
Processed: C00011
Processed: C00012
Processed: C00013
Processed: C00014
Processed: C00015
Processed: C00016
Processed: C00017
Processed: C00018
Processed: C00019
Processed: C00020
Processed: C00021
Processed: C00022
Processed: C00023
Processed: C00024
Processed: C00025
Processed: C00026
Processed: C00027
Processed: C00028
Processed: C00029
Processed: C00030
Processed: C00031
Processed: C00032
Processed: C00033
Processed: C00034
Processed: C00035
Processed: C00036
Processed: C00037
Processed: C00038
Processed: C00039
Processed: C00040
Processed: C00041
Processed: C00042
Processed: C00043
Processed: C00044
Processed: C00045
Processed: C00046
Processed: C00047
Processed: C00048
Processed: C00049
Processed: C00050
Processed: C00051
Processed: C00052
Processed: C00053
Processed: C00054
Processed: C00055
Processed:

In [15]:
recommendations = hybrid_recommend("C00001", n=10)
recommendations

,product_id,product_name,category,brand,price,rating,collaborative_score,content_scores,hybrid_score
5,P000237,Raw Seeds - Sunflower Pumpkin Flax Seeds,Gourmet & World Food,True Elements,166.00,4.4,0.522234,1.000000,0.713340
0,P015536,Belgian Chocolate Cookies,"Bakery, Cakes & Dairy",FabBox,250.00,3.2,1.000000,0.142944,0.657178
12,P009846,Regrowth Hair Oil,Beauty & Hygiene,INATUR,296.00,NaN,0.455532,0.819045,0.600937
9,P012599,Jamnagar Chunda Achar,Snacks & Branded Foods,Graminway,159.00,2.6,0.522234,0.305170,0.435409
3,P004097,Special K,Snacks & Branded Foods,Kelloggs,210.00,4.2,0.579208,0.213032,0.432738
2,P010637,Water Juice Glass Set - Salsa Hi Ball,"Kitchen, Garden & Pets",Ocean,679.00,3.0,0.603122,0.136934,0.416647
1,P026388,"Shower Sponge - Lemon & White, Colour May Vary",Beauty & Hygiene,Panache,159.20,4.0,0.643566,0.040658,0.402403
6,P002553,Celeste Coffret For Women,Beauty & Hygiene,Skinn by Titan,1610.25,4.7,0.522234,0.161743,0.378038
7,P026493,Mealmaker Masala Soya Mini Chunks,"Foodgrains, Oil & Masala",Saffola,45.00,4.5,0.522234,0.128154,0.364602
8,P021849,Noodles - Vegetable,Gourmet & World Food,Koka,94.50,4.3,0.522234,0.088479,0.348732


In [20]:
all_recommendations_df = pd.concat(
    all_recommendations,
    ignore_index=True
)
print("Total recommendation rows:", len(all_recommendations_df))
print("Unique customers:", all_recommendations_df["customer_id"].nunique())
all_recommendations_df.head()

Total recommendation rows: 30000
Unique customers: 3000


,product_id,product_name,category,brand,price,rating,collaborative_score,content_scores,hybrid_score,customer_id
0,P000237,Raw Seeds - Sunflower Pumpkin Flax Seeds,Gourmet & World Food,True Elements,166.0,4.4,0.522234,1.000000,0.713340,C00001
1,P015536,Belgian Chocolate Cookies,"Bakery, Cakes & Dairy",FabBox,250.0,3.2,1.000000,0.142944,0.657178,C00001
2,P009846,Regrowth Hair Oil,Beauty & Hygiene,INATUR,296.0,NaN,0.455532,0.819045,0.600937,C00001
3,P012599,Jamnagar Chunda Achar,Snacks & Branded Foods,Graminway,159.0,2.6,0.522234,0.305170,0.435409,C00001
4,P004097,Special K,Snacks & Branded Foods,Kelloggs,210.0,4.2,0.579208,0.213032,0.432738,C00001


In [16]:
recommendations.to_csv("../models/hybid_recommendations_C00001.csv", index=False)

In [21]:
all_recommendations_df.to_csv(
    "../models/hybrid_recommendations_all_customers.csv",
    index=False
)
print("All-customer hybrid recommendations saved successfully.")

All-customer hybrid recommendations saved successfully.
